#Old frequency branch on StyleGAN dataset

In [ ]:
!pip install -q kagglehub

import kagglehub

path = kagglehub.dataset_download("xhlulu/140k-real-and-fake-faces")
print(path)

Using Colab cache for faster access to the '140k-real-and-fake-faces' dataset.
/kaggle/input/140k-real-and-fake-faces


In [ ]:
import os

DATASET_PATH = "/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/train"

print(DATASET_PATH)
print(os.listdir(DATASET_PATH))

/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/train
['fake', 'real']


In [ ]:
import os

base = "/root/.cache/kagglehub/datasets/xhlulu/140k-real-and-fake-faces/versions/2"

for root, dirs, files in os.walk(base):
    level = root.replace(base, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 3:
        for f in files[:5]:
            print(f"{indent}  {f}")

In [ ]:
import os
import random
import torch

from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from torchvision.datasets import ImageFolder


DATASET_PATH = "/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/train"


train_transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])



full_dataset = ImageFolder(DATASET_PATH, transform=None)

print("Classes:", full_dataset.classes)
print("Class mapping:", full_dataset.class_to_idx)
print("Total available:", len(full_dataset))


random.seed(42)
torch.manual_seed(42)

real_label = full_dataset.class_to_idx["real"]
fake_label = full_dataset.class_to_idx["fake"]

real_indices = [
    i for i, (_, label) in enumerate(full_dataset.samples)
    if label == real_label
]

fake_indices = [
    i for i, (_, label) in enumerate(full_dataset.samples)
    if label == fake_label
]

random.shuffle(real_indices)
random.shuffle(fake_indices)

selected_indices = real_indices[:8000] + fake_indices[:8000]
random.shuffle(selected_indices)

full_dataset.samples = [
    full_dataset.samples[i]
    for i in selected_indices
]

full_dataset.imgs = full_dataset.samples
full_dataset.targets = [
    label for _, label in full_dataset.samples
]

print("\nSelected dataset:")
print("Total:", len(full_dataset))
print(
    "Real:",
    sum(label == real_label for _, label in full_dataset.samples)
)
print(
    "Fake:",
    sum(label == fake_label for _, label in full_dataset.samples)
)


train_size = int(0.7 * len(full_dataset))
val_size = int(0.15 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size

generator = torch.Generator().manual_seed(42)

train_sub, val_sub, test_sub = random_split(
    full_dataset,
    [train_size, val_size, test_size],
    generator=generator
)



class TransformSubset(Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        path, label = self.subset.dataset.samples[
            self.subset.indices[idx]
        ]

        image = self.subset.dataset.loader(path)

        return self.transform(image), label



train_set = TransformSubset(train_sub, train_transform)
val_set = TransformSubset(val_sub, test_transform)
test_set = TransformSubset(test_sub, test_transform)



train_loader = DataLoader(
    train_set,
    batch_size=32,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_set,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

test_loader = DataLoader(
    test_set,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

print(
    f"\nTrain: {len(train_set)}, "
    f"Val: {len(val_set)}, "
    f"Test: {len(test_set)}"
)

Classes: ['fake', 'real']
Class mapping: {'fake': 0, 'real': 1}
Total available: 100000

Selected dataset:
Total: 16000
Real: 8000
Fake: 8000

Train: 11200, Val: 2400, Test: 2400


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split , Dataset
import timm
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt



class FrequencyBranch(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.AdaptiveAvgPool2d((4, 4))
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(64*4*4, 128), nn.ReLU(inplace=True), nn.Dropout(0.3))
    def forward(self, x):
        gray = x.mean(dim=1, keepdim=True)
        fft = torch.fft.fft2(gray)
        fft = torch.fft.fftshift(fft, dim=(-2, -1))
        magnitude = torch.abs(fft)
        magnitude = torch.log1p(magnitude)
        magnitude = (magnitude - magnitude.mean(dim=(-2,-1), keepdim=True)) / (magnitude.std(dim=(-2,-1), keepdim=True) + 1e-6)
        out = self.features(magnitude)
        return self.classifier(out)

class FrequencyOnlyDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.freq_branch = FrequencyBranch()
        self.classifier = nn.Linear(128, 2)
    def forward(self, x):
        return self.classifier(self.freq_branch(x))

model = FrequencyOnlyDetector().cuda()

In [ ]:
num_epochs = 15

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss, correct, total = 0, 0, 0
    for inputs, labels in loader:
        inputs, labels = inputs.cuda(), labels.cuda()
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)
    return running_loss/len(loader), 100.*correct/total

def val_epoch(model, loader, criterion):
    model.eval()
    running_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.cuda(), labels.cuda()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)
    return running_loss/len(loader), 100.*correct/total


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
best_val_acc = 0
train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc = val_epoch(model, val_loader, criterion)
    scheduler.step()

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(f"Epoch {epoch+1}/{num_epochs} | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), '/content/drive/MyDrive/best_model_stylegannochange.pth')
        print(f"  -> Best model saved")

print(f"\nBest validation accuracy: {best_val_acc:.2f}%")

Epoch 1/15 | Train Loss: 0.6894 | Train Acc: 53.40% | Val Loss: 0.6888 | Val Acc: 51.33%
  -> Best model saved
Epoch 2/15 | Train Loss: 0.6854 | Train Acc: 54.92% | Val Loss: 0.6823 | Val Acc: 57.79%
  -> Best model saved
Epoch 3/15 | Train Loss: 0.6808 | Train Acc: 56.39% | Val Loss: 0.6839 | Val Acc: 52.83%
Epoch 4/15 | Train Loss: 0.6781 | Train Acc: 57.17% | Val Loss: 0.6808 | Val Acc: 56.79%
Epoch 5/15 | Train Loss: 0.6734 | Train Acc: 58.11% | Val Loss: 0.6769 | Val Acc: 58.29%
  -> Best model saved
Epoch 6/15 | Train Loss: 0.6724 | Train Acc: 58.30% | Val Loss: 0.6768 | Val Acc: 58.17%
Epoch 7/15 | Train Loss: 0.6707 | Train Acc: 58.55% | Val Loss: 0.6761 | Val Acc: 59.08%
  -> Best model saved
Epoch 8/15 | Train Loss: 0.6720 | Train Acc: 57.91% | Val Loss: 0.6732 | Val Acc: 59.33%
  -> Best model saved
Epoch 9/15 | Train Loss: 0.6705 | Train Acc: 58.62% | Val Loss: 0.6744 | Val Acc: 58.54%
Epoch 10/15 | Train Loss: 0.6689 | Train Acc: 59.33% | Val Loss: 0.6764 | Val Acc: 56.83%